In [9]:
%%file producer.py
from kafka import KafkaProducer
import json, random, time
from datetime import datetime

producer = KafkaProducer(
    bootstrap_servers='broker:9092',
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)

sklepy = ['Warszawa', 'Kraków', 'Gdańsk', 'Wrocław']
kategorie = ['elektronika', 'odzież', 'żywność', 'książki']

def generate_transaction():
    if random.random() < 0.05:
        return {
            'tx_id': f'TX{random.randint(1000,9999)}',
            'user_id': f'u{random.randint(1,20):02d}',
            'amount': round(random.uniform(3000.01, 5000.0), 2),
            'store': random.choice(sklepy),
            'category': 'elektronika',
            'timestamp': datetime.now().isoformat(),
            'hour': random.randint(0, 5)
        }
    else:
        return {
            'tx_id': f'TX{random.randint(1000,9999)}',
            'user_id': f'u{random.randint(1,20):02d}',
            'amount': round(random.uniform(5.0, 3000.0), 2),
            'store': random.choice(sklepy),
            'category': random.choice(kategorie),
            'timestamp': datetime.now().isoformat(),
            'hour': random.randint(6, 23)
        }

for i in range(1000):
    tx = generate_transaction()
    producer.send('transactions', value=tx)

    if tx['hour'] <= 5:
        print(f"[{i+1}] PODEJRZANE {tx['tx_id']} | {tx['amount']:.2f} PLN | {tx['category']} | godz: {tx['hour']}")
    else:
        print(f"[{i+1}] {tx['tx_id']} | {tx['amount']:.2f} PLN | {tx['category']} | {tx['store']} | godz: {tx['hour']}")
    
    time.sleep(0.5)

producer.flush()
producer.close()

Overwriting producer.py


In [10]:
%%file consumer_filter.py
from kafka import KafkaConsumer
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

for message in consumer:
    tx = message.value
    if tx["amount"] > 1000:
        print(f"🚨 ALERT - Duża kwota! {tx['tx_id']} | {tx['amount']} PLN | Sklep: {tx['store']}")

Overwriting consumer_filter.py


In [11]:
%%file consumer_enrich.py
from kafka import KafkaConsumer
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

for message in consumer:
    tx = message.value
    if tx["amount"] > 3000:
        tx["risk_level"] = "HIGH"
    elif tx["amount"] > 1000:
        tx["risk_level"] = "MEDIUM"
    else:
        tx["risk_level"] = "LOW"

    if tx["risk_level"] == "HIGH":
        print(f"HIGHT {tx['tx_id']} | {tx['amount']} PLN | Sklep: {tx['store']}")
    elif tx["risk_level"] == "MEDIUM":
        print(f"MEDIUM {tx['tx_id']} | {tx['amount']} PLN | Sklep: {tx['store']}")
    else:
        print(f"LOW {tx['tx_id']} | {tx['amount']} PLN | Sklep: {tx['store']}")

Overwriting consumer_enrich.py


In [12]:
%%file consumer_count.py
from kafka import KafkaConsumer
from collections import Counter, defaultdict
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    auto_offset_reset='earliest',
    group_id='count-group',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

store_counts = Counter()
total_amount = defaultdict(float)
msg_count = 0

for message in consumer:

    tx = message.value

    store = tx['store']
    amount = tx['amount']

    store_counts[store] += 1
    total_amount[store] += amount
    msg_count += 1

    if msg_count % 10 == 0:

        for s in store_counts:
            count = store_counts[s]
            total = total_amount[s]
            print(f"{s:<15} | {count:<10} | {total:<12.2f}")
        
        print("="*50 + "\n")

Overwriting consumer_count.py


In [15]:
%%file scoring_consumer.py
from kafka import KafkaConsumer, KafkaProducer
import json

def score_transaction(tx):
    score = 0
    rules = []

    store = tx['store']
    category = tx['category']
    amount = tx['amount']
    hour = tx['hour']

    if amount > 3000:
        score += 3
        rules.append('R1')

    if amount > 1500 and category == 'elektronika':
        score += 2
        rules.append('R2')

    if hour < 6:
        score += 2
        rules.append('R3')

    return score, rules

consumer = KafkaConsumer('transactions', bootstrap_servers='broker:9092',
    auto_offset_reset='earliest', group_id='scoring-group',
    value_deserializer=lambda x: json.loads(x.decode('utf-8')))

alert_producer = KafkaProducer(bootstrap_servers='broker:9092',
    value_serializer=lambda v: json.dumps(v).encode('utf-8'))

for message in consumer:
    tx = message.value
    
    score, rules = score_transaction(tx)

    if score >= 3:
        tx['score'] = score
        tx['rules'] = rules
        
        alert_producer.send('alerts', value=tx)

        print(f"ID: {tx['tx_id']} | PKT: {score} | SKLEP: {tx['store']}")



Overwriting scoring_consumer.py
